<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="200"/>
</div>


# AfriCare Support Analytics — Masterclass Pratique
## Notebook · 3 heures · Colab-ready

---

> **Comment utiliser ce notebook :**
> - Les blocs `🎓` expliquent le **pourquoi** — lis-les avant d'exécuter
> - Les cellules `# 📝 TODO` sont à compléter avec ton propre code
> - Les `# 💡 Indice` te guident si tu bloques
> - Les sections `### 🧠 Tes observations` t'invitent à interpréter les résultats
> - Les cellules de **setup** (imports, connexion) sont déjà complètes — exécute-les sans les modifier

| | |
|---|---|
| **Projet** | Customer Support Analytics — AfriCare Services |
| **Durée** | 3 heures |
| **Stack** | Python · DuckDB · scikit-learn · Power BI |
| **Niveau** | Avancé |

### Plan de la masterclass

| Partie | Sujet | Durée estimée |
|---|---|---|
| **0** | Setup & contexte métier | 10 min |
| **1** | Chargement & audit des données | 20 min |
| **2** | Nettoyage & feature engineering | 30 min |
| **3** | SQL Analytics — KPIs & performance | 50 min |
| **4** | Machine Learning — détection de risque | 45 min |
| **5** | Power BI — architecture & DAX | 25 min |

---

# 🧾 La mission

Tu es Data Analyst chez **AfriCare Services**, un opérateur multi-pays (telecom, fintech, e-commerce) présent dans 8 pays d'Afrique et d'Europe.

Le Directeur du Support Client, **M. Eric KOUAME**, t'a confié une mission stratégique pour piloter et améliorer la performance du service client.

---

## Le brief métier

> *"On gère plus de 5 000 tickets par mois et franchement, on perd le fil. Certains clients
> attendent 3 jours pour une réponse qui aurait dû venir en 2 heures. Les SLA explosent
> sur certaines catégories et personne ne comprend pourquoi.*
>
> *Ce dont j'ai besoin : comprendre où sont nos vrais problèmes. Ensuite, un outil qui
> prédit si un ticket est à risque dès sa création. Et en bonus, un dashboard complet
> pour piloter mes 12 agents au quotidien."*

— **M. Eric KOUAME**, Directeur du Support Client, AfriCare Services

> **À toi de jouer — traduis ce brief en 3 attentes analytiques :**

> - *"Comprendre où sont les problèmes"* → quel type d'analyse ? quel notebook ?
> - *"Prédire si un ticket est à risque"* → quel type de modèle ML ?
> - *"Dashboard pour piloter mes agents"* → quel outil ?
>
> *Note tes réponses ci-dessous avant de continuer.*


---
## ⚙️ Partie 0 — Setup `[10 min]`

> Ces cellules sont déjà complètes. Exécute-les dans l'ordre.

In [ ]:
!pip install duckdb jupysql --quiet


In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Patch
import seaborn as sns
import duckdb
import os

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#F9F9F8',
    'axes.grid':        True,
    'grid.alpha':       0.35,
    'font.size':        11,
})

COLORS = {
    'primary':   '#534AB7',
    'secondary': '#1D9E75',
    'warning':   '#EF9F27',
    'danger':    '#E24B4A',
    'neutral':   '#888780',
}

print('✅ Imports OK')


### Chargement des données

> **Option A — Google Drive :** exécute la cellule ci-dessous et monte ton Drive.
> Les CSV doivent être dans : `Mon Drive/DataProjectLab/projects/customer_support/dataset/`

> **Option B — GitHub :** si Drive non disponible, les CSV sont chargés automatiquement depuis GitHub.


---
## 📦 Partie 1 — Chargement & Audit des données `[20 min]`

### Le dictionnaire des données

> 🎓 **Avant de toucher au code, lire toujours le dictionnaire en entier.**
> C'est l'étape la plus sous-estimée par les analystes juniors.

| Table | Lignes | Description |
|---|---|---|
| `tickets.csv` | 15 287 | Table principale — un ticket = une ligne |
| `agents.csv` | 12 | L'équipe support |
| `categories.csv` | variable | Catégories de problèmes avec SLA associé |
| `interactions.csv` | variable | Échanges email/chat par ticket |
| `sla_alerts.csv` | variable | Alertes SLA déclenchées |

**Colonnes clés de `tickets.csv` :**

| Colonne | Description |
|---|---|
| `ticket_id` | Identifiant unique TKT000001 |
| `sla_breach` | ⭐ **Variable cible** : 1 si SLA dépassé, 0 sinon |
| `ratio_sla` | `resolution_heures / sla_heures` — ratio > 1 = breach |
| `first_response_heures` | Délai avant première réponse |
| `resolution_heures` | Durée totale de résolution |
| `csat` | Note satisfaction 1-5 (NULL si pas de retour) |
| `in_backlog` | 1 si ticket en backlog |
| `reopened` | 1 si ticket rouvert |

> 🎓 **Point clé :** `sla_breach` et `ratio_sla` sont liés logiquement.
> Si `ratio_sla > 1` alors `sla_breach` **doit** valoir 1. Vérifier cette cohérence
> est la première vérification à faire dans l'audit.


In [ ]:
# Chargement des données
BASE_URL = "https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/main/projets/customer_support_analytics/data/"


agents     = pd.read_csv(BASE_URL + "agents.csv")
categories = pd.read_csv(BASE_URL + "categories.csv")
tickets    = pd.read_csv(BASE_URL + "tickets.csv", parse_dates=['created_at'])
interactions = pd.read_csv(BASE_URL + "interactions.csv", parse_dates=['timestamp'])
sla_alerts = pd.read_csv(BASE_URL + "sla_alerts.csv", parse_dates=['alert_timestamp'])

# Vérification immédiate des dimensions
print(f"{'Table':<15} {'Lignes':>8} {'Colonnes':>10}")
print("-" * 36)
for name, df in [('tickets', tickets), ('agents', agents), ('categories', categories),
                  ('interactions', interactions), ('sla_alerts', sla_alerts)]:
    print(f"{name:<15} {len(df):>8,} {len(df.columns):>10}")

for name, df in [('tickets', tickets), ('agents', agents), ...]:
    print(f'{name} : {len(df):,} lignes × {len(df.columns)} colonnes')


### Audit qualité rapide

> 🎓 **L'audit avant le nettoyage.** On cherche : nulls, doublons, types incorrects,
> incohérences métier. En 5 minutes, on sait ce qu'il faut corriger.


In [ ]:
# 📝 TODO : Réaliser l'audit qualité en 5 vérifications
# 💡 Indice 1 : Nulls : tickets[col].isnull().sum() sur les colonnes critiques
# 💡 Indice 2 : Doublons : tickets['ticket_id'].duplicated().sum()
# 💡 Indice 3 : Délais négatifs : (tickets['first_response_heures'] < 0).sum()
# 💡 Indice 4 : Incohérence métier : ratio_sla > 1 ET sla_breach == 0 (ou l'inverse)
# 💡 Indice 5 : Taux SLA breach global : tickets['sla_breach'].mean() * 100

cols_critiques = ['sla_breach', 'ratio_sla', 'first_response_heures', 'resolution_heures', 'csat']
# 1. Nulls
for col in cols_critiques:
    n = ...
    print(f'{col} : {n} nulls')

# 2. Doublons
n_dup = ...
print(f'Doublons : {n_dup}')

# 3. Délais négatifs
# ...

# 4. Incohérence sla_breach / ratio_sla
# ...

# 5. Taux breach global
taux_breach = ...
print(f'Taux SLA breach : {taux_breach:.1f}% — objectif < 10%')


### 🧠 Tes observations

> - Combien de nulls y a-t-il sur `csat` ? Est-ce une anomalie ou une valeur normale ?
> - Y a-t-il des incohérences entre `sla_breach` et `ratio_sla` ? Combien ? Laquelle est la source de vérité ?
> - Le taux de SLA breach global est-il acceptable ? Quel est l'écart avec l'objectif de 10% du secteur ?


---
## 🧹 Partie 2 — Nettoyage & Feature Engineering `[30 min]`

> 🎓 **3 règles fondamentales :**
> 1. **Toujours justifier chaque décision** — médiane vs moyenne, left join vs inner join
> 2. **Distinguer anomalie technique et anomalie métier** — délai négatif = bug vs CSAT null = non applicable
> 3. **Définir la variable cible AVANT le feature engineering** pour éviter le data leakage


In [ ]:
# 📝 TODO : Appliquer les 3 corrections de nettoyage essentielles
# 💡 Indice 1 : Correction 1 — Doublons : tickets.drop_duplicates(subset='ticket_id', keep='first')
# 💡 Indice 2 : Correction 2 — Délais négatifs : imputer par la médiane des valeurs >= 0 (pas la moyenne — distribution asymétrique)
# 💡 Indice 3 : Correction 3 — Incohérences sla_breach : corriger sla_breach selon ratio_sla > 1 (source de vérité)
# 💡 Indice 4 : Afficher le nombre de corrections appliquées à chaque étape

# Correction 1 : doublons
n_avant = len(tickets)
tickets = tickets.drop_duplicates(...)
print(f'Doublons supprimés : {n_avant - len(tickets)}')

# Correction 2 : délais négatifs
for col in ['first_response_heures', 'resolution_heures']:
    mask_neg = tickets[col] < 0
    mediane  = ...
    tickets.loc[mask_neg, col] = mediane
    print(f'{mask_neg.sum()} délais négatifs corrigés dans {col}')

# Correction 3 : incohérences sla_breach
mask_inco = ...
tickets.loc[mask_inco, 'sla_breach'] = ...
print(f'Incohérences corrigées : {mask_inco.sum()}')


### Fusion des tables

> 🎓 **Left join, pas inner join.**
> Inner join supprimerait les tickets sans agent ou catégorie connus.
> On veut les analyser — ils peuvent révéler un problème d'affectation.


In [ ]:
# 📝 TODO : Fusionner tickets + agents + catégories avec des LEFT JOINs
# 💡 Indice 1 : tickets.merge(agents[cols_agents], on='agent_id', how='left')
# 💡 Indice 2 : Chaîner un second .merge(..., on='category_id', how='left')
# 💡 Indice 3 : suffixes=('', '_cat') pour éviter les doublons de colonnes 'nom'
# 💡 Indice 4 : Afficher la shape finale et les premières lignes

df = (tickets
      .merge(agents[['agent_id', 'nom', 'tier', 'bureau', 'csat_moyen', 'taux_resolution_pct']],
             on='agent_id', how='left')
      .merge(categories[...], on='category_id', how='left', suffixes=('', '_cat'))
)
print(f'Table fusionnée : {df.shape}')


### Feature Engineering

> 🎓 **L'ordre est impératif :** nettoyage → **variable cible** → features.
> Si une feature encode indirectement la cible après coup, c'est du leakage.


In [ ]:
# 📝 TODO : Créer la variable cible ML et les features temporelles + flags business
# 💡 Indice 1 : Variable cible : ticket_at_risk = 1 si sla_breach=1 OU in_backlog=1 OU statut='Escaladé' OU reopened=1
# 💡 Indice 2 : Combiner avec | (ou logique) et .astype(int)
# 💡 Indice 3 : Features temporelles depuis created_at : heure_creation, jour_semaine, mois, est_weekend, est_heure_creuse
# 💡 Indice 4 : est_weekend = jour_semaine >= 5 | est_heure_creuse = heure < 8 ou heure >= 20
# 💡 Indice 5 : Flags business : is_priorite_haute = priorite <= 2

# Variable cible
df['ticket_at_risk'] = (
    (df['sla_breach'] == 1) |
    (df['in_backlog'] == 1) |
    ...
).astype(int)
print(f'ticket_at_risk=1 : {df["ticket_at_risk"].mean()*100:.1f}%')

# Features temporelles
df['heure_creation']   = df['created_at'].dt.hour
df['jour_semaine']     = ...
df['mois']             = ...
df['est_weekend']      = ...
df['est_heure_creuse'] = ...

# Flags business
df['is_priorite_haute'] = ...


### 🧠 Tes observations

> - Quel est le taux de `ticket_at_risk = 1` ? Est-ce équilibré ? Quelle stratégie ML cela implique-t-il ?
> - À quelle heure de la journée y a-t-il le plus de tickets créés ? Correspond-il aux heures de pointe attendues ?
> - Pourquoi définit-on `ticket_at_risk` avec un OU logique (plusieurs conditions) et non uniquement `sla_breach` ?


In [ ]:
# 📝 TODO : Exporter le dataset nettoyé en CSV
# 💡 Indice 1 : df.to_csv('support_clean_analytics.csv', index=False)
# 💡 Indice 2 : Vérifier la bonne écriture en relisant les 3 premières lignes avec pd.read_csv(..., nrows=3)
# 💡 Indice 3 : Afficher : nb lignes, nb colonnes, taux ticket_at_risk, période couverte

df.to_csv('support_clean_analytics.csv', index=False)
print(f'✅ {len(df):,} lignes × {df.shape[1]} colonnes exportées')
print(f'Taux ticket_at_risk : {df["ticket_at_risk"].mean()*100:.1f}%')


---
## 🔎 Partie 3 — SQL Analytics : KPIs & Performance `[50 min]`

> 🎓 **Pourquoi DuckDB ?** Portable, aucune installation serveur, lit les CSV directement.
> Syntaxe proche de PostgreSQL/SQL Server.

**Patterns SQL à maîtriser dans cette partie :**
- `AVG(CASE WHEN cond THEN 1.0 ELSE 0.0 END) * 100` — calcul de taux
- `RANK() OVER (ORDER BY ...)` — classement des agents
- `LAG(col) OVER (ORDER BY ...)` — comparaison période précédente
- `WITH ... AS (CTE)` — requête en deux étapes lisibles
- `CASE WHEN` dans `SELECT` — binning (tranches de délai)


In [ ]:
# Connexion DuckDB — cellule déjà complète, exécute-la
import duckdb, os
con = duckdb.connect()
con.execute("""
    CREATE TABLE tickets      AS SELECT * FROM read_csv_auto('support_clean_analytics.csv');
    CREATE TABLE agents       AS SELECT * FROM read_csv_auto('agents.csv');
    CREATE TABLE categories   AS SELECT * FROM read_csv_auto('categories.csv');
    CREATE TABLE interactions AS SELECT * FROM read_csv_auto('interactions.csv');
    CREATE TABLE sla_alerts   AS SELECT * FROM read_csv_auto('sla_alerts.csv');
""")
n = con.execute('SELECT COUNT(*) FROM tickets').fetchone()[0]
print(f'✅ {n:,} tickets chargés dans DuckDB')

# ── Activation de JupySQL pour écrire %%sql directement dans les cellules ──
%load_ext sql
%sql con --alias duckdb
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
print('%%sql prêt ✅')


### 3.1 — KPIs globaux opérationnels

> 🎓 **Tout calculer en une seule passe.** Une requête → une lecture de table.
> **Le pattern taux en SQL :**
> ```sql
> AVG(CASE WHEN sla_breach=1 THEN 1.0 ELSE 0.0 END) * 100
> ```
> `CASE WHEN` transforme un booléen en 0/1, `AVG` calcule la proportion, `* 100` donne le %.


In [ ]:
%%sql df_kpi <<
-- 📝 TODO : Calculer tous les KPIs globaux en une seule passe
-- 💡 Indice 1 : AVG(CASE WHEN sla_breach=1 THEN 1.0 ELSE 0.0 END)*100 pour les taux
-- 💡 Indice 2 : ROUND(..., 1) pour les taux | ROUND(..., 2) pour les moyennes
-- 💡 Indice 3 : AVG(CASE WHEN csat IS NOT NULL THEN csat END) pour ignorer les nulls CSAT
-- 💡 Indice 4 : Calculer : taux_sla_breach_pct, taux_resolution_pct, moy_first_response_h,
--               moy_resolution_h, csat_moyen, taux_backlog_pct, taux_escalade_pct
SELECT
    COUNT(*)                                                              AS total_tickets
    -- Compléter les autres KPIs...
FROM tickets


In [ ]:
# Affichage des KPIs après exécution du %%sql ci-dessus
print('=== KPIs GLOBAUX ===')
for col, val in df_kpi.iloc[0].items():
    print(f'  {col:<30} : {val}')

# 📝 TODO : Ajouter la ligne de contexte pour M. Kouame
# 💡 Indice : breach = df_kpi['taux_sla_breach_pct'].iloc[0]
#             print(f'Taux breach : {breach}% — objectif < 10% — écart : +{breach-10:.1f} pts')


### 🧠 Tes observations

> - Le taux SLA breach est-il acceptable ? Quel est l'écart avec l'objectif de 10% du secteur ?
> - Quel KPI te surprend le plus ? Que révèle-t-il sur le fonctionnement de l'équipe support ?
> - Le CSAT moyen est-il satisfaisant ? En dessous de quel seuil estimes-tu que c'est problématique ?


### 3.2 — SLA par catégorie (TOP critiques)

> 🎓 **JOIN enrichi.** On joint `tickets` avec `categories` pour avoir le nom et le SLA.
> Sans le JOIN, on ne verrait que des `category_id` illisibles pour M. Kouame.


In [ ]:
%%sql df_sla_cat <<
-- 📝 TODO : Calculer le taux SLA breach par catégorie
-- 💡 Indice 1 : JOIN categories c ON t.category_id = c.category_id
-- 💡 Indice 2 : GROUP BY c.nom, c.sla_heures | ORDER BY taux_breach DESC LIMIT 8
-- 💡 Indice 3 : AVG(CASE WHEN t.sla_breach=1 THEN t.resolution_heures - c.sla_heures END)
--               pour le dépassement moyen (ELSE NULL implicite → exclut les non-breach)
SELECT
    c.nom           AS categorie,
    c.sla_heures,
    COUNT(*)        AS nb_tickets
    -- Compléter : taux_breach, resolution_moy_h, depassement_moy_h
FROM tickets t
JOIN categories c ON t.category_id = c.category_id
GROUP BY c.nom, c.sla_heures
ORDER BY taux_breach DESC
LIMIT 8


In [ ]:
# Affichage du tableau
print(df_sla_cat.to_string(index=False))

# 📝 TODO : Construire le graphique barres horizontales
# 💡 Indice 1 : fig, ax = plt.subplots(figsize=(11, 5))
# 💡 Indice 2 : Palette sémantique — rouge si > 50%, orange si > 25%, vert sinon
# 💡 Indice 3 : ax.barh(df_sla_cat['categorie'], df_sla_cat['taux_breach'], color=colors)
# 💡 Indice 4 : ax.axvline(10, ...) pour la ligne de référence objectif 10%

fig, ax = plt.subplots(figsize=(11, 5))
colors = ...  # Compléter
# ...
plt.tight_layout(); plt.show()

# Insight automatique
cat_crit = df_sla_cat.iloc[0]
print(f"\n💡 Catégorie critique : '{cat_crit['categorie']}'")
print(f"   {cat_crit['taux_breach']}% de breach — SLA : {cat_crit['sla_heures']:.0f}h")


### 🧠 Tes observations

> - Quelle catégorie a le taux de breach le plus élevé ? Est-ce logique avec la nature des problèmes qu'elle traite ?
> - Le SLA de cette catégorie est-il adapté à la réalité des délais ? Quelle recommandation ferais-tu à M. Kouame ?


### 3.3 — Performance des agents avec RANK()

> 🎓 **RANK() OVER (ORDER BY ...)** classe les lignes sans en supprimer.
> Rang 1 = meilleur agent (moins de breach si ORDER BY ASC).
> `NULLS LAST` : les agents sans CSAT passent en dernière position.


In [ ]:
%%sql df_agt <<
-- 📝 TODO : Classer les agents avec RANK() — taux breach ET CSAT
-- 💡 Indice 1 : JOIN agents a ON t.agent_id = a.agent_id | GROUP BY a.nom, a.tier, a.bureau
-- 💡 Indice 2 : RANK() OVER (ORDER BY AVG(sla_breach) ASC) → rang 1 = meilleur (peu de breach)
-- 💡 Indice 3 : RANK() OVER (ORDER BY AVG(csat) DESC NULLS LAST) → rang 1 = meilleur CSAT
-- 💡 Indice 4 : NULLS LAST évite qu'un agent sans CSAT soit classé premier
SELECT
    a.nom, a.tier, a.bureau,
    COUNT(*) AS nb_tickets
    -- Compléter : taux_breach, csat_reel, resolution_moy_h, rang_breach, rang_csat
FROM tickets t
JOIN agents a ON t.agent_id = a.agent_id
GROUP BY a.nom, a.tier, a.bureau
ORDER BY taux_breach ASC


In [ ]:
print(df_agt[['nom','tier','nb_tickets','taux_breach','csat_reel',
               'rang_breach','rang_csat']].to_string(index=False))

# 📝 TODO : Calculer et afficher l'insight écart de performance
# 💡 Indice : ecart = df_agt['taux_breach'].max() - df_agt['taux_breach'].min()
#             best = df_agt.iloc[0] | worst = df_agt.iloc[-1]


### 🧠 Tes observations

> - Quel est l'écart de taux breach entre le meilleur et le pire agent ? Est-il surprenant ?
> - Le rang breach et le rang CSAT sont-ils corrélés ? Un agent avec peu de breach a-t-il forcément un bon CSAT ?
> - Quelle recommandation concrète ferais-tu à M. Kouame pour les agents les moins performants ?


### 3.4 — Évolution mensuelle avec LAG()

> 🎓 **LAG(col) OVER (ORDER BY mois)** retourne la valeur du mois précédent.
> `taux_breach - LAG(taux_breach)` → négatif = amélioration, positif = dégradation.
> On utilise une **CTE** (WITH ... AS) pour séparer le calcul en deux étapes lisibles.


In [ ]:
%%sql df_mensuel <<
-- 📝 TODO : Tendance mensuelle avec CTE + LAG()
-- 💡 Indice 1 : CTE mensuel — GROUP BY strftime(created_at, '%Y-%m')
--               avec COUNT(*), AVG(CASE WHEN sla_breach=1 ...) AS taux_breach
-- 💡 Indice 2 : Requête principale — LAG(taux_breach) OVER (ORDER BY mois) AS breach_prev
-- 💡 Indice 3 : ROUND(taux_breach - LAG(taux_breach) OVER (ORDER BY mois), 1) AS evolution_breach
-- 💡 Indice 4 : CASE WHEN taux_breach < LAG(...) THEN '↘ Amélioration' ... END AS tendance_mois
WITH mensuel AS (
    SELECT
        strftime(created_at, '%Y-%m')  AS mois,
        COUNT(*)                        AS nb_tickets
        -- Compléter : taux_breach, csat_moy
    FROM tickets
    GROUP BY strftime(created_at, '%Y-%m')
)
SELECT
    mois, nb_tickets
    -- Compléter : taux_breach, breach_prev, evolution_breach, tendance_mois
FROM mensuel
ORDER BY mois


In [ ]:
# 📝 TODO : Graphique double axe (volume barres + breach courbe)
# 💡 Indice 1 : fig, ax1 = plt.subplots(figsize=(13, 5)) | ax2 = ax1.twinx()
# 💡 Indice 2 : ax1.bar(x, df_mensuel['nb_tickets'], ...) pour le volume
# 💡 Indice 3 : ax2.plot(x, df_mensuel['taux_breach'], ...) pour le breach
# 💡 Indice 4 : ax2.axhline(10, ...) pour la ligne objectif 10%

fig, ax1 = plt.subplots(figsize=(13, 5))
ax2 = ax1.twinx()
x = range(len(df_mensuel))
# ...
plt.tight_layout(); plt.show()

# Tendance globale
tendance = df_mensuel['taux_breach'].iloc[-3:].mean() - df_mensuel['taux_breach'].iloc[:3].mean()
print(f'Tendance globale : {tendance:+.1f} pts (positif = dégradation)')


### 🧠 Tes observations

> - La tendance du taux breach est-elle croissante, stable ou décroissante sur la période ?
> - Y a-t-il des mois avec une forte dégradation soudaine ? À quoi pourrait-on l'attribuer ?
> - M. Kouame attendait une amélioration naturelle — les données la confirment-elles ?


### 3.5 — CSAT vs Délai de résolution

> 🎓 **C'est le graphique le plus impactant pour M. Kouame.**
> Il prouve statistiquement que l'opérationnel impacte la satisfaction.
> Ce n'est pas une intuition — c'est dans les données.

> **Pattern CASE WHEN pour créer des tranches (binning) :**
> ```sql
> CASE WHEN resolution_heures <= 4 THEN '1 - < 4h'
>      WHEN resolution_heures <= 12 THEN '2 - 4-12h'
>      ...
> END AS tranche_delai
> ```
> Préfixe numérique (`'1 - '`, `'2 - '`) pour forcer l'ordre chronologique dans le tri.


In [ ]:
%%sql df_csat_delai <<
-- 📝 TODO : CSAT moyen par tranche de délai de résolution
-- 💡 Indice 1 : 5 tranches — <= 4h | 4-12h | 12-24h | 1-3j | > 3j
-- 💡 Indice 2 : Préfixer : '1 - < 4h', '2 - 4-12h'... pour forcer l'ordre chronologique
-- 💡 Indice 3 : WHERE csat IS NOT NULL avant toute agrégation
-- 💡 Indice 4 : ORDER BY tranche_delai — le préfixe numérique garantit l'ordre chronologique
SELECT
    CASE
        WHEN resolution_heures <= 4  THEN '1 - < 4h'
        -- Compléter les tranches suivantes...
        ELSE '5 - > 3j'
    END                              AS tranche_delai,
    ROUND(AVG(csat), 2)              AS csat_moy,
    COUNT(*)                         AS nb_avis
    -- Bonus : ajouter pct_promoteurs et pct_detracteurs
FROM tickets
WHERE csat IS NOT NULL
GROUP BY tranche_delai
ORDER BY tranche_delai


In [ ]:
print(df_csat_delai.to_string(index=False))

# 📝 TODO : Courbe CSAT moyen par tranche + fill_between
# 💡 Indice 1 : labels = [t.split(' - ')[1] for t in df_csat_delai['tranche_delai']]
# 💡 Indice 2 : ax.plot(labels, df_csat_delai['csat_moy'], marker='o', linewidth=2.5)
# 💡 Indice 3 : ax.fill_between(labels, 1, df_csat_delai['csat_moy'], alpha=0.1)
# 💡 Indice 4 : ax.annotate(f'{v:.2f}', (i, v), xytext=(0,10), textcoords='offset points')

labels = [t.split(' - ')[1] for t in df_csat_delai['tranche_delai']]
fig, ax = plt.subplots(figsize=(9, 5))
# ...
plt.tight_layout(); plt.show()

c_rapide = df_csat_delai.iloc[0]['csat_moy']
c_lent   = df_csat_delai.iloc[-1]['csat_moy']
print(f'Impact délai : < 4h → CSAT {c_rapide:.2f}/5 | > 3j → CSAT {c_lent:.2f}/5')
print(f'Écart        : {c_rapide - c_lent:.2f} points de satisfaction')


### 🧠 Tes observations

> - Quelle est la différence de CSAT entre un ticket résolu en < 4h et un ticket résolu en > 3j ?
> - En dessous de quel délai le CSAT dépasse-t-il 4.0/5 ? Quel objectif opérationnel cela implique-t-il ?
> - Comment formules-tu cet insight dans un email à M. Kouame ? (Chiffre précis → signification métier → action)


In [ ]:
# 📝 TODO : Rédiger la synthèse analytique pour M. Kouame
# 💡 Indice 1 : Format — chiffre précis → signification métier → recommandation concrète
# 💡 Indice 2 : Insight 1 — df_kpi.iloc[0]['taux_sla_breach_pct']
# 💡 Indice 3 : Insight 2 — df_sla_cat.iloc[0] (catégorie la plus critique)
# 💡 Indice 4 : Insight 3 — df_agt (meilleur vs pire : df_agt.iloc[0] / df_agt.iloc[-1])
# 💡 Indice 5 : Insight 4 — c_rapide et c_lent depuis df_csat_delai

print('=' * 65)
print('  SYNTHÈSE ANALYTIQUE — AFRICAIRE SUPPORT')
print('=' * 65)

kv = df_kpi.iloc[0]
print(f'\nINSIGHT 1 — SLA en crise')
print(f'  {kv["taux_sla_breach_pct"]:.1f}% de breach — objectif 10%')
print(f'  → Recommandation : ...')

# Compléter INSIGHT 2, 3, 4...


---
## 🤖 Partie 4 — Machine Learning : Détection de tickets à risque `[45 min]`

> 🎓 **5 règles ML à graver dans ta tête avant de coder :**
> 1. **Définir la variable cible avant le feature engineering** — c'est ce que le modèle prédit
> 2. **Interdire les features disponibles seulement après la résolution** — data leakage
> 3. **Coupure temporelle** — jamais de split aléatoire sur des séries temporelles
> 4. **Traiter le déséquilibre des classes** — `class_weight='balanced'`
> 5. **Optimiser le seuil** — 0.5 est rarement le seuil optimal en production


### 4.1 — Sélection des features — La règle du leakage

> 🎓 **Features INTERDITES** (disponibles seulement après la résolution) :
> `resolution_heures`, `sla_breach`, `ratio_sla`, `csat`, `statut`, `nb_contacts`, `reopened`

> **Features AUTORISÉES** (disponibles dès la création du ticket) :
> `heure_creation`, `jour_semaine`, `est_weekend`, `priorite`, `canal`, `pays`,
> `tier`, `bureau`, `csat_moyen` (historique agent), `taux_resolution_pct` (historique agent),
> `sla_heures` (défini avant la création)


In [ ]:
from sklearn.ensemble          import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model      import LogisticRegression
from sklearn.metrics           import (classification_report, confusion_matrix,
                                        roc_auc_score, roc_curve, f1_score,
                                        precision_score, recall_score, precision_recall_curve)
from sklearn.preprocessing     import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection   import TimeSeriesSplit, cross_val_score
print('✅ Librairies ML chargées')


In [ ]:
# 📝 TODO : Définir les listes FEATURES_CAT, FEATURES_NUM et appliquer le LabelEncoder sur les catégorielles
# 💡 Indice 1 : FEATURES_CAT = variables catégorielles à encoder : ['canal', 'pays', 'tier', 'bureau']
# 💡 Indice 2 : FEATURES_NUM = variables numériques directement utilisables (sans encode)
# 💡 Indice 3 : LabelEncoder pour les catégorielles — adapté aux arbres, pas aux modèles linéaires
# 💡 Indice 4 : Créer une colonne col+'_enc' pour chaque variable de FEATURES_CAT
# 💡 Indice 5 : FEATURES = FEATURES_NUM + [c+'_enc' for c in FEATURES_CAT]

df_ml = pd.read_csv('support_clean_analytics.csv', parse_dates=['created_at'])
df_ml = df_ml.sort_values('created_at').reset_index(drop=True)

FEATURES_CAT = ['canal', 'pays', 'tier', 'bureau']
FEATURES_NUM = [
    'heure_creation', 'jour_semaine', 'mois', 'est_weekend', 'est_heure_creuse',
    'priorite', 'is_priorite_haute', 'sla_heures',
    'csat_moyen', 'taux_resolution_pct',
]
TARGET = 'ticket_at_risk'

# Encodage LabelEncoder
for col in FEATURES_CAT:
    if col in df_ml.columns:
        le = LabelEncoder()
        df_ml[col + '_enc'] = le.fit_transform(...)

FEATURES = FEATURES_NUM + [c + '_enc' for c in FEATURES_CAT if c in df_ml.columns]
FEATURES = [f for f in FEATURES if f in df_ml.columns]
print(f'{len(FEATURES)} features retenues')


### 4.2 — Coupure temporelle

> 🎓 **Jamais `train_test_split` avec `shuffle=True` sur des séries temporelles.**
> On entraînerait sur des tickets de décembre pour prédire des tickets de janvier.
> La coupure temporelle imite les conditions réelles de production.

> ```
> |← 80% TRAIN (passé) →|← 20% TEST (futur) →|
> ```


In [ ]:
# 📝 TODO : Créer le split temporel 80/20 en respectant l'ordre chronologique
# 💡 Indice 1 : df_ml est déjà trié par created_at — utiliser les indices, pas random
# 💡 Indice 2 : n_train = int(len(df_ml) * 0.80)
# 💡 Indice 3 : train = df_ml.iloc[:n_train] | test = df_ml.iloc[n_train:]
# 💡 Indice 4 : Afficher les plages de dates train et test pour confirmer la coupure

n_total = len(df_ml)
n_train = int(n_total * 0.80)

train = df_ml.iloc[:n_train].copy()
test  = df_ml.iloc[n_train:].copy()

X_train = train[FEATURES].fillna(0)
y_train = train[TARGET]
X_test  = test[FEATURES].fillna(0)
y_test  = test[TARGET]

print(f'Train : {len(train):,} tickets ({train["created_at"].min().date()} → {train["created_at"].max().date()})')
print(f'Test  : {len(test):,} tickets ({test["created_at"].min().date()} → {test["created_at"].max().date()})')
print(f'Taux risque train : {y_train.mean()*100:.1f}% | test : {y_test.mean()*100:.1f}%')


### 🧠 Tes observations

> - Le taux de `ticket_at_risk = 1` est-il similaire dans train et test ? Si très différent, qu'est-ce que cela signifie ?
> - Pourquoi un split aléatoire donnerait-il de meilleures métriques mais un modèle inutile en production ?


### 4.3 — Entraînement et comparaison de 3 modèles

> 🎓 **Toujours établir une baseline.** Si RF ne fait pas mieux que LR,
> la complexité supplémentaire ne se justifie pas.

> **`class_weight='balanced'`** donne plus de poids aux tickets à risque.
> Sans ça : le modèle prédit tout 'normal' → Recall = 0% sur les risques.

> **`compute_sample_weight`** pour GradientBoosting : ce modèle n'a pas `class_weight`
> en paramètre, on passe les poids dans `.fit(sample_weight=sw)`.


In [ ]:
# 📝 TODO : Entraîner les 3 modèles (LR, RF, GB) avec class_weight ou sample_weight
# 💡 Indice 1 : LR : LogisticRegression(class_weight='balanced', max_iter=500, random_state=42)
# 💡 Indice 2 : RF : RandomForestClassifier(n_estimators=200, max_depth=8, class_weight='balanced', random_state=42)
# 💡 Indice 3 : GB : GradientBoostingClassifier(n_estimators=150, max_depth=4, learning_rate=0.1, random_state=42)
# 💡 Indice 4 : Pour GB : sw = compute_sample_weight('balanced', y_train) puis gb.fit(X_train, y_train, sample_weight=sw)
# 💡 Indice 5 : Pour chaque modèle : predict_proba(X_test)[:, 1] pour les probabilités

# Modèle 1 : Logistic Regression (baseline)
lr = LogisticRegression(...)
lr.fit(X_train, y_train)
y_prob_lr = lr.predict_proba(X_test)[:, 1]
y_pred_lr = lr.predict(X_test)

# Modèle 2 : Random Forest
rf = RandomForestClassifier(...)
rf.fit(X_train, y_train)
y_prob_rf = ...
y_pred_rf = ...

# Modèle 3 : Gradient Boosting
sw = compute_sample_weight('balanced', y_train)
gb = GradientBoostingClassifier(...)
gb.fit(X_train, y_train, sample_weight=sw)
y_prob_gb = ...
y_pred_gb = ...

print('3 modèles entraînés ✅')


In [ ]:
# 📝 TODO : Construire le tableau comparatif AUC / F1 / Recall / Precision pour les 3 modèles
# 💡 Indice 1 : roc_auc_score(y_test, y_prob) — indépendant du seuil
# 💡 Indice 2 : f1_score(y_test, y_pred) | recall_score(...) | precision_score(...)
# 💡 Indice 3 : Créer un DataFrame avec une ligne par modèle
# 💡 Indice 4 : Identifier le meilleur modèle selon l'AUC

results = {}
for nom, y_pred, y_prob in [
    ('Logistic Regression', y_pred_lr, y_prob_lr),
    ('Random Forest',       y_pred_rf, y_prob_rf),
    ('Gradient Boosting',   y_pred_gb, y_prob_gb),
]:
    results[nom] = {
        'AUC':    round(roc_auc_score(y_test, y_prob), 3),
        'F1':     round(f1_score(y_test, y_pred), 3),
        'Recall': round(recall_score(y_test, y_pred), 3),
        'Precision': round(precision_score(y_test, y_pred), 3),
    }

df_results = pd.DataFrame(results).T
print(df_results.to_string())
print(f'Meilleur modèle AUC : {df_results["AUC"].idxmax()}')


### 🧠 Tes observations

> - Quel modèle a le meilleur AUC ? Est-ce surprenant ?
> - Le Recall est-il satisfaisant ? Rappelle-toi : un faible Recall = beaucoup de vrais risques NON détectés (FN).
> - Quelle est la différence entre AUC et F1-Score ? Dans quel cas privilégie-t-on l'un sur l'autre ?


In [ ]:
# 📝 TODO : Tracer la matrice de confusion et les courbes ROC pour les 3 modèles (figure 1×2)
# 💡 Indice 1 : confusion_matrix(y_test, y_pred_rf) + sns.heatmap pour visualiser
# 💡 Indice 2 : roc_curve(y_test, y_prob) pour chaque modèle sur le même axes[1]
# 💡 Indice 3 : Annoter la matrice : TP, FP, FN (à minimiser), TN
# 💡 Indice 4 : Ajouter la diagonale aléatoire : ax.plot([0,1],[0,1],'k--')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Normal', 'Risque'],
            yticklabels=['Normal', 'Risque'])
axes[0].set_title('Matrice de confusion — Random Forest')
tn, fp, fn, tp = cm.ravel()
print(f'TP={tp} | FP={fp} | FN={fn} (à minimiser) | TN={tn}')

# Courbes ROC
for nom, y_prob, color in [('LR', y_prob_lr, '...'), ...]:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    axes[1].plot(fpr, tpr, color=color, label=f'{nom} (AUC={auc:.3f})')
axes[1].legend(); plt.tight_layout(); plt.show()


### 🧠 Tes observations

> - Combien de tickets à risque (FN) n'ont pas été détectés ? Quel est l'impact pour M. Kouame ?
> - Préfères-tu un modèle avec plus de FP ou plus de FN dans le contexte du support client ? Justifie.
> - La courbe ROC du Random Forest est-elle significativement meilleure que la LR baseline ?


### 4.4 — Optimisation du seuil

> 🎓 **Le seuil 0.5 est rarement optimal en production.**
> On cherche le seuil qui maximise le Recall (détecter un maximum de vrais risques)
> tout en maintenant une Precision acceptable (limiter les fausses alarmes).
> **Contrainte métier : Recall > 0.75** — mieux vaut over-alerter que manquer un risque.


In [ ]:
# 📝 TODO : Trouver le seuil optimal sur y_prob_rf : Recall > 0.75, F1 maximisé
# 💡 Indice 1 : Boucle sur np.arange(0.1, 0.9, 0.02)
# 💡 Indice 2 : y_pred_s = (y_prob_rf >= seuil).astype(int)
# 💡 Indice 3 : Condition : recall_score(y_test, y_pred_s) > 0.75
# 💡 Indice 4 : Parmi les seuils valides, garder celui avec le meilleur F1
# 💡 Indice 5 : Tracer la courbe Precision vs Recall avec precision_recall_curve

seuil_optimal = 0.5
best_f1 = 0
for seuil in np.arange(0.1, 0.9, 0.02):
    y_pred_s = (y_prob_rf >= seuil).astype(int)
    if y_pred_s.sum() == 0: continue
    rec = recall_score(y_test, y_pred_s)
    f1  = f1_score(y_test, y_pred_s)
    if rec > 0.75 and f1 > best_f1:
        best_f1 = f1
        seuil_optimal = round(seuil, 2)

y_pred_final = (y_prob_rf >= seuil_optimal).astype(int)
print(f'Seuil optimal : {seuil_optimal}')
print(f'Recall : {recall_score(y_test, y_pred_final):.3f}')
print(f'F1     : {f1_score(y_test, y_pred_final):.3f}')


### 🧠 Tes observations

> - Quel seuil as-tu trouvé ? Est-il plus bas ou plus haut que 0.5 ? Explique pourquoi.
> - En abaissant le seuil, le Recall augmente mais la Precision baisse — pourquoi ce trade-off ?
> - M. Kouame peut ajuster ce seuil après quelques semaines de production — quand faudrait-il le remonter ?


In [ ]:
# 📝 TODO : Exporter le fichier d'alertes tickets_risque_scores.csv avec 4 niveaux d'alerte
# 💡 Indice 1 : Fonction niveau_alerte(score) : ROUGE > 0.70, ORANGE > 0.45, JAUNE > 0.25, VERT sinon
# 💡 Indice 2 : Ajouter score_risque = y_prob_rf et niveau_alerte dans df_test
# 💡 Indice 3 : Trier par score_risque décroissant (les plus urgents en premier)
# 💡 Indice 4 : Exporter avec encoding='utf-8-sig' pour la compatibilité Excel/Power BI
# 💡 Indice 5 : Afficher la distribution des niveaux + taux vrais positifs par niveau

def niveau_alerte(score):
    if score > 0.70: return 'ROUGE — Intervention immédiate'
    if score > 0.45: return 'ORANGE — Surveiller'
    if score > 0.25: return 'JAUNE — Attention'
    return 'VERT — Normal'

df_test_out = test.copy()
df_test_out['score_risque']  = y_prob_rf
df_test_out['niveau_alerte'] = df_test_out['score_risque'].apply(niveau_alerte)
df_test_out['predit_risque'] = (df_test_out['score_risque'] >= seuil_optimal).astype(int)

cols_out = ['ticket_id', 'created_at', 'canal', 'priorite', 'statut',
            'score_risque', 'niveau_alerte', 'predit_risque', 'ticket_at_risk']
df_alertes = df_test_out[cols_out].sort_values('score_risque', ascending=False)
df_alertes.to_csv('tickets_risque_scores.csv', index=False, encoding='utf-8-sig')
print(df_alertes['niveau_alerte'].value_counts().to_string())


### 🧠 Tes observations

> - Combien de tickets sont classés ROUGE ? Combien un superviseur doit-il traiter chaque matin ?
> - Le taux de vrais positifs est-il plus élevé pour les niveaux ROUGE et ORANGE ? Qu'est-ce que cela valide ?
> - Les seuils ROUGE/ORANGE/JAUNE sont des paramètres métier — M. Kouame peut les ajuster. Comment déciderais-tu de les modifier après 1 mois en production ?


In [ ]:
# 📝 TODO : Valider la stabilité du modèle avec TimeSeriesSplit (5 folds) et visualiser les scores
# 💡 Indice 1 : tscv = TimeSeriesSplit(n_splits=5)
# 💡 Indice 2 : cross_val_score(rf_cv, X_all, y_all, cv=tscv, scoring='roc_auc')
# 💡 Indice 3 : Si std > 0.05 : modèle instable → retraining périodique conseillé
# 💡 Indice 4 : Graphique courbe AUC et F1 sur les 5 folds + ligne pointillée seuil 0.75

tscv  = TimeSeriesSplit(n_splits=5)
X_all = df_ml[FEATURES].fillna(0)
y_all = df_ml[TARGET]

rf_cv      = RandomForestClassifier(n_estimators=200, max_depth=8, class_weight='balanced', random_state=42)
scores_auc = cross_val_score(rf_cv, X_all, y_all, cv=tscv, scoring='roc_auc')
scores_f1  = cross_val_score(rf_cv, X_all, y_all, cv=tscv, scoring='f1')

# Graphique + affichage résultats
print(f'AUC moyen : {scores_auc.mean():.3f} ± {scores_auc.std():.3f}')
print(f'Stabilité : {"✅ STABLE" if scores_auc.std() < 0.05 else "⚠️ VARIABLE"}')


### 🧠 Tes observations

> - L'écart-type de l'AUC est-il inférieur à 0.05 ? Le modèle est-il stable dans le temps ?
> - Les scores s'améliorent-ils sur les folds récents (plus de données) ou se dégradent-ils ?
> - À quelle fréquence recommandes-tu de ré-entraîner le modèle ? Mensuel, trimestriel, annuel ? Justifie.


---
## 📊 Partie 5 — Power BI : Architecture & DAX `[25 min]`

> 🎓 **Un dashboard n'est pas un rapport.**
> Un rapport affiche des chiffres. Un dashboard déclenche des décisions.
> La différence tient en une question : en 30 secondes, un superviseur sait-il quoi faire ?

### Architecture 5 pages

| Page | Question business | Public | Fréquence |
|---|---|---|---|
| 1 — Alertes & Risques | Quels tickets nécessitent une intervention aujourd'hui ? | Superviseur | Chaque matin |
| 2 — Performance SLA | Où et quand le SLA explose-t-il ? | Manager | Hebdo |
| 3 — Performance Agents | Qui performe ? Qui est surchargé ? | Manager | Hebdo |
| 4 — Tendances & Backlog | Quelle est la tendance globale ? | Direction | Mensuel |
| 5 — Satisfaction CSAT | Quel lien entre délai et satisfaction ? | Direction | Mensuel |

> 🎓 **La règle des 4-6 visuels par page.** Au-delà, l'œil ne sait plus où regarder.
> Une page = une question business = un public cible.


### 📝 TODO DAX — Mesures à compléter dans Power BI

#### Table Calendrier (Modélisation → Nouvelle table)

```dax
Calendrier =
ADDCOLUMNS(
    CALENDAR(MIN(tickets[created_at]), MAX(tickets[created_at])),
    "Annee",      YEAR([Date]),
    "Mois",       MONTH([Date]),
    "NomMois",    FORMAT([Date], "MMM YYYY"),
    "JourSemaine",WEEKDAY([Date], 2),
    "EstWeekend", IF(WEEKDAY([Date],2)>=6, 1, 0)
)
```

#### Mesure 1 — Taux SLA Breach % *(à compléter)*

```dax
Taux SLA Breach % =
DIVIDE(
    CALCULATE(COUNTROWS(tickets), tickets[sla_breach] = ___),  -- valeur de breach
    COUNTROWS(tickets),
    0
) * ___  -- multiplier par 100 pour obtenir un %
```

#### Mesure 2 — CSAT Moyen *(à compléter)*

```dax
CSAT Moyen =
CALCULATE(
    AVERAGEX(tickets, tickets[csat]),
    ___(tickets[csat])  -- filter pour exclure les nulls
)
```

#### Mesure 3 — Tickets ROUGE *(à compléter)*

```dax
Tickets ROUGE =
CALCULATE(
    COUNTROWS(tickets_risque_scores),
    SEARCH("___", tickets_risque_scores[niveau_alerte], 1, 0) > 0  -- mot à chercher
)
```

#### Mesure 4 — Bandeau alerte conditionnel *(à compléter)*

```dax
Couleur Alerte =
IF(
    [Tickets ROUGE] > 0,
    "___",   -- code couleur rouge hex sans #
    "___"    -- code couleur vert hex sans #
)
```

#### Mesure 5 — Évolution breach vs mois précédent *(à compléter)*

```dax
Breach Mois Précédent =
CALCULATE(
    [Taux SLA Breach %],
    DATEADD(Calendrier[Date], ___, MONTH)  -- décalage en mois
)

Evolution Breach = [Taux SLA Breach %] - [___]  -- mesure à référencer
```


### 🧠 Tes observations

> - La mesure `Couleur Alerte` retourne un code hex — dans quel paramètre Power BI l'utilises-tu pour changer dynamiquement la couleur d'une carte ?
> - Quelle est la différence entre `COUNTROWS` et `COUNT` en DAX ? Quand utilises-tu l'un vs l'autre ?
> - Pourquoi la Table Calendrier est-elle indispensable pour les analyses temporelles (LAG, DATEADD) en Power BI ?


### Check-list avant livraison

```
☐  Modèle en étoile créé (vérifier les relations)
☐  Table Calendrier liée sur created_at
☐  Taux SLA breach cohérent avec SQL Partie 3
☐  Bandeau alerte Page 1 : rouge si Tickets ROUGE > 0, vert sinon
☐  5 boutons de navigation entre pages
☐  Slicers : Période | Canal | Pays | Agent
☐  tickets_risque_scores.csv chargé comme source ML
```

> 🎓 **Valider les KPIs Power BI vs SQL :** le taux SLA breach affiché dans Power BI
> doit correspondre exactement au résultat de ta requête SQL en Partie 3.
> Toute divergence signale une erreur dans le modèle de données ou les mesures DAX.


---

## ✅ Bilan de la masterclass — À compléter

| Partie | Livrable produit | Valeur obtenue |
|---|---|---|
| 1 — Audit | Rapport qualité | Taux breach = ___% |
| 2 — Nettoyage | `support_clean_analytics.csv` | ___ lignes · ___ colonnes |
| 3 — SQL | KPIs + insights M. Kouame | Catégorie critique : ___ |
| 4 — ML | `tickets_risque_scores.csv` | AUC = ___ · Seuil = ___ |
| 5 — Power BI | Dashboard 5 pages | Mesures DAX : ___ |

### Ta recommandation finale à M. Kouame

> *Rédige ici en 3-4 phrases ta recommandation principale.*
> *Format : chiffre précis → signification métier → action concrète → résultat attendu.*

> ...


---

**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.
